In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 100)

In [3]:
df = pd.read_csv("../data/JohnWayneFlights_2026-03-14.csv")
print(df.shape)
print(df.head())
df.info()

(191, 10)
       id flight_type              city   status gate  claim       scheduled_time  \
0  708656     arrival          San Jose  Arrived   19    5.0  2026-03-14 08:25:00   
1  708657     arrival           Oakland  Arrived   18    5.0  2026-03-14 08:30:00   
2      -1     arrival  Dallas-Ft. Worth  Arrived    7    1.0  2026-03-14 08:33:00   
3      -1     arrival     San Francisco  Arrived    9    4.0  2026-03-14 08:43:00   
4      -1     arrival           Phoenix  Arrived   15    6.0  2026-03-14 08:55:00   

           actual_time    flight_codes route_airport_code  
0  2026-03-14 08:12:00          WN4455                SJC  
1  2026-03-14 08:14:00           WN726                OAK  
2  2026-03-14 08:06:00          AA1686                DFW  
3  2026-03-14 08:22:00  UA5937, AC4246                SFO  
4  2026-03-14 08:55:00          F91719                PHX  
<class 'pandas.DataFrame'>
RangeIndex: 191 entries, 0 to 190
Data columns (total 10 columns):
 #   Column              

In [5]:
#### Standardize Columns ###
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

df.columns.tolist()

['id',
 'flight_type',
 'city',
 'status',
 'gate',
 'claim',
 'scheduled_time',
 'actual_time',
 'flight_codes',
 'route_airport_code']

In [6]:
### inspect missing values ###
df.isna().sum()

id                     0
flight_type            0
city                   0
status                 0
gate                   2
claim                 83
scheduled_time         0
actual_time            0
flight_codes           0
route_airport_code     0
dtype: int64

In [10]:
### convert time columns into real date times ###
df["scheduled_time"] = pd.to_datetime(df["scheduled_time"], errors = "coerce")
df["actual_time"] = pd.to_datetime(df["actual_time"], errors = "coerce")

display(df[["scheduled_time", "actual_time"]].head())
df[["scheduled_time", "actual_time"]].info()

,scheduled_time,actual_time
0,2026-03-14 08:25:00,2026-03-14 08:12:00
1,2026-03-14 08:30:00,2026-03-14 08:14:00
2,2026-03-14 08:33:00,2026-03-14 08:06:00
3,2026-03-14 08:43:00,2026-03-14 08:22:00
4,2026-03-14 08:55:00,2026-03-14 08:55:00


<class 'pandas.DataFrame'>
RangeIndex: 191 entries, 0 to 190
Data columns (total 2 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   scheduled_time  191 non-null    datetime64[us]
 1   actual_time     191 non-null    datetime64[us]
dtypes: datetime64[us](2)
memory usage: 3.1 KB


In [ ]:
### Delay row in minutes ###
### Calculation ###
df["delay_min"] = (
    (df["actual_time"] - df["scheduled_time"]).dt.total_seconds()/60
)

df[["scheduled_time", "actual_time", "delay_min"]]
### How to interpret ###
#negative  -> early
#0 -> on time
#positive -> late 

,scheduled_time,actual_time,delay_min
0,2026-03-14 08:25:00,2026-03-14 08:12:00,-13.0
1,2026-03-14 08:30:00,2026-03-14 08:14:00,-16.0
2,2026-03-14 08:33:00,2026-03-14 08:06:00,-27.0
3,2026-03-14 08:43:00,2026-03-14 08:22:00,-21.0
4,2026-03-14 08:55:00,2026-03-14 08:55:00,0.0
...,...,...,...
186,2026-03-14 19:59:00,2026-03-14 20:25:00,26.0
187,2026-03-14 20:10:00,2026-03-14 20:15:00,5.0
188,2026-03-14 20:43:00,2026-03-14 20:43:00,0.0
189,2026-03-14 21:10:00,2026-03-14 21:10:00,0.0


In [ ]:
### Clean ID ###
# Assuming -1 is missing ID -> turned to nan
# we don't want fake values pretending to be ID
df["id"] = pd.to_numeric(df["id"], errors="coerce")
df["id"] = df["id"].replace(-1,np.nan)

display(df["id"].head(10))
df["id"].isna().sum()

0    708656.0
1    708657.0
2         NaN
3         NaN
4         NaN
5         NaN
6         NaN
7    708666.0
8         NaN
9    708670.0
Name: id, dtype: float64

np.int64(160)

In [ ]:
### Clean Claim ###
# Make the claim numeric where possible, missing where not applicable
# Useful since most departures don't have a baggauage claim
df["claim"] = df["claim"].replace("N/A", np.nan)
df["claim"] = pd.to_numeric(df["claim"], errors="coerce")

display(df["claim"].head(10))
df["claim"].value_counts(dropna=False).sort_index()


0    5.0
1    5.0
2    1.0
3    4.0
4    6.0
5    1.0
6    2.0
7    5.0
8    3.0
9    5.0
Name: claim, dtype: float64

claim
1.0    19
2.0    16
3.0    21
4.0    16
5.0    25
6.0    10
7.0     1
NaN    83
Name: count, dtype: int64

In [15]:
### Clean gate ###
# standardize the gate turning 1a -> 1A 22b -> 22B
df["gate"] = df["gate"].astype(str).str.strip().str.upper()
df["gate"].head()

0    19
1    18
2     7
3     9
4    15
Name: gate, dtype: str

In [ ]:
### Clean Status ###
# Avoids weird variants caused by casing or spaces
df["status"] = df["status"].astype(str).str.strip().str.title()
sorted(df["status"].unique())

['Arrived', 'Cancelled', 'Delayed', 'Departed', 'Early', 'On Time']

In [ ]:
### Extract the main flight code ###
# We want one main flight code per row for airline analysis
df["primary_flight_code"] = df["flight_codes"].astype(str).str.split(",").str[0].str.strip()
df[["flight_codes", "primary_flight_code"]]

,flight_codes,primary_flight_code
0,WN4455,WN4455
1,WN726,WN726
2,AA1686,AA1686
3,"UA5937, AC4246",UA5937
4,F91719,F91719
...,...,...
186,AS336,AS336
187,WN2805,WN2805
188,AS2116,AS2116
189,WN4524,WN4524


In [19]:
### Extract airline code and flight number ###
df["airline_code"] = df["primary_flight_code"].str.extract(r"^([A-Z]+)")
df["flight_number"] = df["primary_flight_code"].str.extract(r"(\d+)$")

df[["primary_flight_code", "airline_code", "flight_number"]].head(15)

,primary_flight_code,airline_code,flight_number
0,WN4455,WN,4455
1,WN726,WN,726
2,AA1686,AA,1686
3,UA5937,UA,5937
4,F91719,F,91719
5,AA543,AA,543
6,DL1198,DL,1198
7,WN574,WN,574
8,AS3440,AS,3440
9,WN643,WN,643


In [21]:
### Map airline codes to airline names ###
airline_map = {
    "WN": "Southwest",
    "AA": "American",
    "UA": "United",
    "AC": "Air Canada",
    "DL": "Delta",
    "AS": "Alaska",
    "NK": "Spirit",
    "F9": "Frontier",
    "G4": "Allegiant",
    "MX": "Breeze",
    "WS": "WestJet"
}
df["airline"] = df["airline_code"].map(airline_map).fillna("Other")
df[["primary_flight_code", "airline_code", "airline"]].head(20)
df["airline"].value_counts()

airline
Alaska        42
American      38
United        34
Southwest     31
Delta         22
Other          8
Spirit         6
Breeze         6
Air Canada     2
WestJet        2
Name: count, dtype: int64

In [23]:
### Normalize Flights status ###
def normalize_status(row):
    status = row["status"]
    delay = row["delay_min"]

    if status == "Cancelled":
        return "Cancelled"
    if pd.isna(delay):
        return "Unknown"
    if delay <= -15:
        return "Early"
    if delay <= 15:
        return "On Time"
    return "Delayed"
### Perforce label based on timing ###
df["status_clean"] = df.apply(normalize_status, axis=1)
display(df[["flight_type", "status", "delay_min", "status_clean"]].head(20))
df["status_clean"].value_counts()

,flight_type,status,delay_min,status_clean
0,arrival,Arrived,-13.0,On Time
1,arrival,Arrived,-16.0,Early
2,arrival,Arrived,-27.0,Early
3,arrival,Arrived,-21.0,Early
4,arrival,Arrived,0.0,On Time
5,arrival,Arrived,-16.0,Early
6,arrival,Arrived,-6.0,On Time
7,arrival,Arrived,18.0,Delayed
8,arrival,Arrived,-18.0,Early
9,arrival,Arrived,-12.0,On Time


status_clean
On Time      125
Early         36
Delayed       27
Cancelled      3
Name: count, dtype: int64

In [25]:
### Cancelled flights should not contribute to average delay ###
# Prevents cancelled flights from messing up delay stats
df.loc[df["status_clean"] == "Cancelled", "delay_min"] = np.nan

In [ ]:
### create date/time features ###
# USE -> delays by hour, traffic by day, trend charts, PowerBI filtering
df["flight_date"] = df["scheduled_time"].dt.time
df["hour"] = df["scheduled_time"].dt.hour
df["day_of_week"] = df["scheduled_time"].dt.day.name
df["month"] = df["scheduled_time"].dt.month

display(df[["flight_type", "status", "delay_min", "status_clean"]].head(20))
df["status_clean"].value_counts()

,flight_type,status,delay_min,status_clean
0,arrival,Arrived,-13.0,On Time
1,arrival,Arrived,-16.0,Early
2,arrival,Arrived,-27.0,Early
3,arrival,Arrived,-21.0,Early
4,arrival,Arrived,0.0,On Time
5,arrival,Arrived,-16.0,Early
6,arrival,Arrived,-6.0,On Time
7,arrival,Arrived,18.0,Delayed
8,arrival,Arrived,-18.0,Early
9,arrival,Arrived,-12.0,On Time


status_clean
On Time      125
Early         36
Delayed       27
Cancelled      3
Name: count, dtype: int64

In [29]:
### Create KPI flag columns ###
# Cancellation rate, delay rate, on-time rate
df["is_cancelled"]= (df["status_clean"] == "Cancelled").astype(int)
df["is_delayed"]= (df["status_clean"] == "Delayed").astype(int)
df["is_ontime"]= (df["status_clean"] == "On Time").astype(int)
df["is_early"]= (df["status_clean"] == "Early").astype(int)

df["is_cancelled"]= (df["status_clean"] == "Cancelled").astype(int)
df[["status_clean", "is_cancelled", "is_delayed", "is_ontime", "is_early"]].head(15)

,status_clean,is_cancelled,is_delayed,is_ontime,is_early
0,On Time,0,0,1,0
1,Early,0,0,0,1
2,Early,0,0,0,1
3,Early,0,0,0,1
4,On Time,0,0,1,0
5,Early,0,0,0,1
6,On Time,0,0,1,0
7,Delayed,0,1,0,0
8,Early,0,0,0,1
9,On Time,0,0,1,0


In [30]:
### Performance within 15 minutes ###
df["otp_15_flag"] = np.where(
    df["status_clean"] != "Cancelled",
    (df["delay_min"] <= 15).astype(int),
    np.nan
)
df[["delay_min", "status_clean", "otp_15_flag"]].head(20)

,delay_min,status_clean,otp_15_flag
0,-13.0,On Time,1.0
1,-16.0,Early,1.0
2,-27.0,Early,1.0
3,-21.0,Early,1.0
4,0.0,On Time,1.0
5,-16.0,Early,1.0
6,-6.0,On Time,1.0
7,18.0,Delayed,0.0
8,-18.0,Early,1.0
9,-12.0,On Time,1.0


In [31]:
### Quality Checks ###
print("Rows:", len(df))
print("Missing IDs:", df["id"].isna().sum())
print("Missing claims:", df["claim"].isna().sum())
print("Missing scheduled_time:", df["scheduled_time"].isna().sum())
print("Missing actual_time:", df["actual_time"].isna().sum())

print("\nStatus clean counts:")
print(df["status_clean"].value_counts())

print("\nFlight type counts:")
print(df["flight_type"].value_counts())

print("\nAirline counts:")
print(df["airline"].value_counts())

Rows: 191
Missing IDs: 160
Missing claims: 83
Missing scheduled_time: 0
Missing actual_time: 0

Status clean counts:
status_clean
On Time      125
Early         36
Delayed       27
Cancelled      3
Name: count, dtype: int64

Flight type counts:
flight_type
arrival      107
departure     84
Name: count, dtype: int64

Airline counts:
airline
Alaska        42
American      38
United        34
Southwest     31
Delta         22
Other          8
Spirit         6
Breeze         6
Air Canada     2
WestJet        2
Name: count, dtype: int64


In [ ]:
### Inspect Unusually Large Delays ###
df.sort_values("delay_min", ascending=False)[
    ["flight_type", "city", "airline", "scheduled_time", "actual_time", "delay_min", "status_clean"]
].head(10)

,flight_type,city,airline,scheduled_time,actual_time,delay_min,status_clean
20,arrival,Las Vegas,Southwest,2026-03-14 10:30:00,2026-03-14 13:35:00,185.0,Delayed
159,departure,Denver,United,2026-03-14 13:27:00,2026-03-14 16:24:00,177.0,Delayed
53,arrival,Las Vegas,Southwest,2026-03-14 15:15:00,2026-03-14 17:13:00,118.0,Delayed
154,departure,Newark,United,2026-03-14 12:48:00,2026-03-14 14:16:00,88.0,Delayed
48,arrival,Las Vegas,Spirit,2026-03-14 13:58:00,2026-03-14 15:23:00,85.0,Delayed
167,departure,Las Vegas,Spirit,2026-03-14 14:53:00,2026-03-14 16:09:00,76.0,Delayed
31,arrival,Chicago/O'Hare,United,2026-03-14 11:51:00,2026-03-14 13:02:00,71.0,Delayed
61,arrival,San Jose Del Cabo,Southwest,2026-03-14 16:25:00,2026-03-14 17:19:00,54.0,Delayed
96,arrival,Oakland,Southwest,2026-03-14 21:15:00,2026-03-14 22:01:00,46.0,Delayed
99,arrival,Chicago/O'Hare,American,2026-03-14 21:39:00,2026-03-14 22:24:00,45.0,Delayed


In [34]:
df.to_csv("../data/processed/jwa_flights_clean.csv", index=False)

##### QUESTIONS THAT CAN BE ANSWERED
###### -------------------------------------------------------------------------
###### 1. On-Time Performance (OTP)
###### q1 -> What percentage of flights arrived or departed within 15 minutes of schedule?
###### 2. Average Delay
###### q2 -> How Many Minutes Late Are Flights On Average?
###### 3. Cancellation Rate 
###### q3 -> What percentage of flights were cancelled?
###### 4. Delay Rate
###### q4 -> What Percentage Of flights experienced delays?
###### 5. Early Arrival Rate 
###### q5 -> How Many flights arrive early 
###### -------------------------------------------------------------------------